# Judge answer pairs in both orders (Colab, T4 GPU)

[Open in Colab](https://colab.research.google.com/github/khacbac/nlp/blob/main/experiment/judge.ipynb)

Qwen2.5-7B-Instruct judges each of the 1500 answer pairs twice, once with each answer shown first,
and saves the verdict with the probabilities of "A", "B" and "C". Results go to
`MyDrive/nlp-answers/judgments_vi.jsonl`, one line per judgment.

- *Runtime > Change runtime type > T4 GPU*, then run all cells.
- **After a disconnect, run all cells again**: finished judgments are skipped.
- Expect about 5 min install + 30-60 min judging.

## 1. Drive + code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = "/content/drive/MyDrive/nlp-answers"   # also holds answers_*.jsonl
REPO = "/content/nlp"

import os
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} fetch -q origin && git -C {REPO} reset -q --hard origin/main
else:
    !git clone -q https://github.com/khacbac/nlp.git {REPO}
%cd {REPO}/experiment
!git log -1 --format="code at %h: %s"

## 2. GPU + vLLM

In [ ]:
import subprocess
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then run from cell 1 again.")
!pip install -q vllm

## 3. Trial: 2 questions (12 judgments)
Check that each judgment ends with `[[A]]`, `[[B]]` or `[[C]]` and that `p_x` is filled in.

In [ ]:
!python judge.py --limit 2 --answers-dir {OUT_DIR} --out-dir {OUT_DIR}

import json
for line in open(f"{OUT_DIR}/judgments_vi.jsonl", encoding="utf-8").readlines()[:4]:
    r = json.loads(line)
    print(r["qid"], r["order"], "verdict:", r["verdict"], "winner:", r["winner"],
          "p_x:", None if r["p_x"] is None else round(r["p_x"], 3))
    print(r["judgment"][-300:], "\n")

## 4. All 500 questions

In [ ]:
!python judge.py --answers-dir {OUT_DIR} --out-dir {OUT_DIR}
!wc -l {OUT_DIR}/judgments_vi.jsonl

## Next
Expect 3000 lines. Download `judgments_vi.jsonl` into `experiment/data/` and run
`python analyze.py` (see README). The file names the model of each answer; do not share it with annotators.